> **Hands-on workshop for the AI-Driven DevOps (AIDD) program, School of Computer Science and Engineering, Jain (Deemed-to-be University), Bengaluru**

# 09 · Observability: traces, spans and the cost of an answer

**CampusMate v6 (agent you can debug)**  ·  ⏱️ Time budget: **15 min theory + 30 min hands-on**

*All student, course, policy and event data in this workshop is fictional.*

## Why this matters

When CampusMate gives a wrong answer in front of 40 students, 'it's AI, it's random' is not an explanation. **Observability** lets you replay what happened: which tool ran with which input, how long each step took, and how many tokens (rupees) it cost. AgentCore sends this to Amazon CloudWatch automatically.

### What you'll do
- Learn traces vs spans vs logs vs metrics
- Find the trace of your own chat
- Use the bad-answer recipe and trace it to the step that caused it
- See where the cost of an answer comes from

## Settings

✏️ Change `STUDENT_ID` to your own ID before running anything else.

In [ ]:
# Settings for this lab
import sys, json, time, uuid, pathlib, subprocess, shutil
import boto3

# ✏️ Put YOUR student ID here (the same one in every notebook). Lower-case letters and digits, e.g. priya07
STUDENT_ID = "student01"

REGION = "us-east-1"
MODEL_ID = "us.amazon.nova-2-lite-v1:0"         # Amazon Nova 2 Lite
CHEAP_MODEL_ID = "us.amazon.nova-2-lite-v1:0"
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA_DIR = ROOT / "data"

PROJECT = f"campusmate{STUDENT_ID}"                    # your AgentCore project (created in notebook 05)
PROJECT_DIR = ROOT / PROJECT                           # the folder it lives in
AGENT_FILE = PROJECT_DIR / "app" / "agent" / "main.py" # your agent's code

def agentcore(*args, folder=None):
    """Run an AgentCore CLI command (the same one you would type in a terminal) and show its output."""
    process = subprocess.Popen([shutil.which("agentcore") or "agentcore", *args], cwd=folder or PROJECT_DIR,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        if "[SDK info]" not in line:
            print(line, end="")
    if process.wait() != 0:
        raise RuntimeError("The agentcore command failed. Read the messages above.")

print("Student:", STUDENT_ID, "| Project folder:", PROJECT_DIR)

## What you built in earlier labs

The AgentCore CLI knows everything in your project. We ask it, so there are no IDs to copy.

In [ ]:
def deployed():
    """Ask the AgentCore CLI what your project has in AWS: {name: details}."""
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "status", "--json"], cwd=PROJECT_DIR, capture_output=True, text=True)
    return {r["name"]: r for r in json.loads(out.stdout)["resources"]}

def ask(prompt, session_id, student="AIDD-DEMO-001"):
    """Send one message to your deployed agent (agentcore invoke) and return its answer."""
    payload = json.dumps({"prompt": prompt, "student_id": student})
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "invoke", "--json", "--runtime", "agent",
                          "--session-id", session_id, "--payload", payload], cwd=PROJECT_DIR, capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(out.stdout + out.stderr)
    return json.loads(json.loads(out.stdout)["body"])["result"]

resources = deployed()
AGENT_ARN = resources["agent"]["arn"]
RUNTIME_ID = AGENT_ARN.split("/")[-1]
LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{RUNTIME_ID}-DEFAULT"      # your agent's logs and traces
for name, r in resources.items():
    print(f"{r['resourceType']:14s} {name:16s} {r.get('arn', '')}")

## Four words

- **Trace**: the whole story of one request (one question → one answer).
- **Span**: one step inside it (the agent, each model call, each tool call).
- **Log**: a line of text written by your code or by AWS.
- **Metric**: a number over time (requests, latency, errors).

🏫 A trace is a student's whole exam day; spans are the stops (gate, hall-ticket check, seat, answer sheet); logs are the invigilator's notes;
metrics are the exam cell's daily counts.

`agentcore deploy` switched tracing on for you (CloudWatch **Transaction Search**). Spans go to your agent's log group, printed above as `LOG_GROUP`.

## 🔬 Live check: make a trace you can find

In [ ]:
good_session = f"lab09-good-{uuid.uuid4()}"
print(ask("Am I eligible for the AIDD303 exam?", good_session, "AIDD-DEMO-003"))
print("\nLog group:", LOG_GROUP)

## The bad-answer recipe 🧪

Ask something **ambiguous**: *"Can I skip class on Friday for the hackathon? Just apply leave for it."* The right path is **on-duty (OD)**
under the hackathon rules, but an agent often reaches for the **leave** rules instead. Let's see what *your* agent picks.

In [ ]:
bad_session = f"lab09-bad-{uuid.uuid4()}"
print(ask("Can I skip class on Friday for the HackAIDD hackathon? Just apply leave for it.", bad_session, "AIDD-DEMO-008"))

## Read the spans with CloudWatch Logs Insights

Spans take 30-90 seconds to arrive, so the cell retries.

In [ ]:
logs = boto3.client("logs", region_name=REGION)
QUERY = """fields @timestamp, name, durationNano / 1000000 as ms,
       `attributes.gen_ai.usage.input_tokens` as tokens_in, `attributes.gen_ai.usage.output_tokens` as tokens_out,
       `attributes.gen_ai.tool.name` as tool
| filter @logStream = 'spans' and `attributes.session.id` = '{session}'
| filter name like /invoke_agent|execute_tool/ or name = 'chat'
| sort @timestamp asc"""

def spans_for(session, tries=10):
    for _ in range(tries):
        q = logs.start_query(logGroupName=LOG_GROUP, startTime=int(time.time()) - 3600, endTime=int(time.time()) + 60,
                             queryString=QUERY.format(session=session))["queryId"]
        while (r := logs.get_query_results(queryId=q))["status"] in ("Running", "Scheduled"):
            time.sleep(1)
        rows = [{c["field"]: c["value"] for c in row} for row in r["results"]]
        if rows:
            return rows
        print("  no spans yet, waiting 15 s..."); time.sleep(15)
    return []

rows = spans_for(bad_session)
for f in rows:
    print(f"{f.get('name', '')[:34]:34s} {float(f.get('ms', 0)):7.0f} ms   tokens in/out = {f.get('tokens_in', '-'):>5}/{f.get('tokens_out', '-'):<5} {f.get('tool', '')}")

## Where the cost comes from

Each `chat` span is one model call. Tokens **in** include the system prompt, the tool descriptions and the whole conversation so far,
so they grow with every step. Output tokens cost 5x more each, but there are fewer of them.

In [ ]:
chats = [f for f in rows if f.get("name") == "chat"]
tokens_in = sum(int(float(f.get("tokens_in") or 0)) for f in chats)
tokens_out = sum(int(float(f.get("tokens_out") or 0)) for f in chats)
cost = tokens_in * 0.33 / 1_000_000 + tokens_out * 2.75 / 1_000_000      # Nova 2 Lite prices, Nov 2026
print(f"{len(chats)} model calls | {tokens_in} tokens in + {tokens_out} out | about ${cost:.4f} (Rs. {cost * 84:.2f})")

## See it in the console

CloudWatch → **GenAI Observability** → **Bedrock AgentCore** → your agent `campusmate<your id>_default_agent` → **Sessions** →
pick the `lab09-bad-...` session → **Trace**. Click the `execute_tool` span to see the exact input the model sent.

## ✍️ Your turn

Diagnose the bad answer: which tool was chosen, which step was slowest, and what would you change (the tool's docstring? the system prompt?).

> The cell below runs safely as-is (placeholders are skipped). Replace each `# TODO` and run it again.

In [ ]:
# YOUR TURN
MY_DIAGNOSIS = {"tool_chosen": "TODO", "slowest_step": "TODO", "fix_idea": "TODO"}
if "TODO" not in MY_DIAGNOSIS.values():
    print(MY_DIAGNOSIS)
    # TODO: try the other recipe: ask("How much is the fee and when is it due?", ...) and read its spans
else:
    print("Skipped: fill MY_DIAGNOSIS first.")

## ✅ Checkpoint

You are on track if:
- You found spans for your `lab09-bad-...` session: `invoke_agent`, `chat`, and at least one `execute_tool`
- You can say which tool the agent picked for the hackathon question

**Stuck or behind?** Copy from `scripts/09_observability.py` and keep going. That's what it's there for.

## 🛠️ Common errors

| You see | Why | Fix |
|---|---|---|
| No spans after 3 minutes | The very first spans in an account can take ~10 minutes | Wait and re-run the cell |
| `MalformedQueryException` | Typo in the query | Keep the backticks around names with dots |
| `ResourceNotFoundException` for the log group | The agent hasn't been called yet | Run the live check first |